# Data Preparation
Nettoyage, transformation et feature engineering basés sur les observations des notebooks 01. 02 et 03.

## Imports et chargement

In [39]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
from scipy.stats import permutation_test
from sklearn.utils import resample

df = pd.read_csv("../data/processed/dataset_combined.csv")
df["DateHeure"] = pd.to_datetime(df["DateHeure"])
df = df.sort_values(["NomVille", "DateHeure"]).reset_index(drop=True)

print(f"Dataset chargé : {df.shape[0]} lignes. {df.shape[1]} colonnes")

Dataset chargé : 10544 lignes. 19 colonnes


## 1. Suppression des doublons

In [40]:
before = len(df)
df = df.drop_duplicates(subset=["NomVille", "DateHeure"], keep="last")
print(f"Doublons supprimés : {before - len(df)}")
print(f"Lignes restantes   : {len(df)}")

Doublons supprimés : 0
Lignes restantes   : 10544


### Observations - Doublons
> Le dataset combiné ne contient aucun doublon. La fusion n'a pas introduit de lignes dupliquées, ce qui confirme la qualité de l'alignement temporel et de la jointure entre les deux sources.

## 2. Correction de structure

In [41]:
# Standardisation des types numériques
numeric_cols = [
    "Temperature",
    "Humidite",
    "Pression",
    "VitesseVent",
    "AqiGlobal",
    "PM25",
    "PM10",
    "NO2",
    "O3",
    "wind_direction_10m",
    "cloud_cover",
    "precipitation",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Heure"] = df["Heure"].astype(int)
df["Mois"] = df["Mois"].astype(int)

# Vérification des valeurs physiquement impossibles
df.loc[df["Humidite"] > 100, "Humidite"] = np.nan
df.loc[df["Humidite"] < 0, "Humidite"] = np.nan
df.loc[df["AqiGlobal"] < 0, "AqiGlobal"] = np.nan

print("Correction de structure terminée.")
print(df.dtypes)

Correction de structure terminée.
NomVille                         str
DateHeure             datetime64[us]
Heure                          int64
Mois                           int64
Temperature                  float64
Humidite                     float64
Pression                       int64
VitesseVent                  float64
AqiGlobal                    float64
PM25                         float64
PM10                         float64
NO2                          float64
O3                           float64
MeteoStatus                      str
AirStatus                        str
wind_direction_10m             int64
cloud_cover                    int64
precipitation                float64
IsWeekend                       bool
dtype: object


### Observations - Structure
> Le dataset combiné est bien structuré. Les colonnes sont correctement nommées et typées. Aucune transformation majeure n'est nécessaire à ce stade.

## 3. Gestion des valeurs manquantes

In [42]:
# Analyse avant traitement
null_pct = df.isnull().sum() / len(df) * 100
print("Taux de NULL avant traitement :")
print(null_pct[null_pct > 0].sort_values(ascending=False).round(2))

Taux de NULL avant traitement :
O3           16.43
PM25          8.16
AqiGlobal     0.17
PM10          0.02
NO2           0.01
dtype: float64


In [43]:
# Suppression des lignes sans variable cible
before = len(df)
df = df.dropna(subset=["AqiGlobal"])
print(f"Lignes supprimées (AqiGlobal NULL) : {before - len(df)}")

# Imputation ciblée
df["precipitation"] = df["precipitation"].fillna(0)

# PM25. O3. NO2. PM10 conservés avec NULL car ils ont une forte corrélation avec AqiGlobal.
# Random Forest et XGBoost les gèrent nativement

print()
print("Taux de NULL après traitement :")
null_pct_after = df.isnull().sum() / len(df) * 100
print(null_pct_after[null_pct_after > 0].sort_values(ascending=False).round(2))

Lignes supprimées (AqiGlobal NULL) : 18

Taux de NULL après traitement :
O3      16.34
PM25     8.06
PM10     0.01
dtype: float64


### Observations - Valeurs manquantes
- 18 lignes supprimées car sans variable cible AqiGlobal. Ce sont des pannes ponctuelles d'AQICN documentées dans le notebook 01 (0.17% du dataset). La perte est négligeable.
- Après traitement le dataset contient 10 526 lignes.
- O3 reste à 16.34% et PM25 à 8.06% de NULL. Ces valeurs sont conservées intentionnellement. Lyon sans capteur PM25 ni O3 et Franconville sans capteur O3 expliquent ces taux. PM10 descend à 0.01% une seule valeur manquante restante. négligeable.
- precipitation est à 0% car le fillna(0) a tout comblé. Temperature. Humidite. Pression. VitesseVent et les trois variables Open-Meteo sont à 0% NULL. parfait.
- **Décision confirmée**: PM25. O3. PM10 et NO2 restent dans le dataset avec leurs NULL. Random Forest et XGBoost les gèrent nativement sans imputation.

## 4. Gestion des valeurs aberrantes

In [44]:
# Détection IQR
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(
        f"{col} : {len(outliers)} outliers (borne inf={lower:.2f} / borne sup={upper:.2f})"
    )
    return lower, upper


print("Détection des outliers (IQR)")
for col in ["AqiGlobal", "PM25", "PM10"]:
    detect_outliers_iqr(df, col)

Détection des outliers (IQR)
AqiGlobal : 34 outliers (borne inf=-10.50 / borne sup=89.50)
PM25 : 23 outliers (borne inf=-28.50 / borne sup=103.50)
PM10 : 506 outliers (borne inf=-10.00 / borne sup=38.00)


In [45]:
# Validation par Bootstrapping
def bootstrap_mean(series, n=1000):
    means = [resample(series.dropna()).mean() for _ in range(n)]
    ci = np.percentile(means, [2.5, 97.5])
    print(f"IC 95% bootstrappé : [{ci[0]:.2f}. {ci[1]:.2f}]")
    return ci


print("Bootstrapping AqiGlobal")
ci_with = bootstrap_mean(df["AqiGlobal"])
ci_without = bootstrap_mean(df[df["AqiGlobal"] <= 100]["AqiGlobal"])
print(f"Avec outliers    : [{ci_with[0]:.2f}. {ci_with[1]:.2f}]")
print(f"Sans outliers    : [{ci_without[0]:.2f}. {ci_without[1]:.2f}]")
print("Les outliers sont influents si les IC diffèrent significativement.")

Bootstrapping AqiGlobal
IC 95% bootstrappé : [38.82. 39.53]
IC 95% bootstrappé : [38.49. 39.12]
Avec outliers    : [38.82. 39.53]
Sans outliers    : [38.49. 39.12]
Les outliers sont influents si les IC diffèrent significativement.


In [46]:
# Test de permutation : AQI mars vs juin
mars = df[df["Mois"] == 3]["AqiGlobal"].dropna().values
juin = df[df["Mois"] == 6]["AqiGlobal"].dropna().values

result = permutation_test(
    (mars, juin),
    lambda x, y: np.mean(x) - np.mean(y),
    permutation_type="independent",
    n_resamples=1000,
)
print(f"p-value mars vs juin : {result.pvalue:.4f}")
print("p-value < 0.05 → différence significative → Mois est une feature pertinente.")

p-value mars vs juin : 0.0020
p-value < 0.05 → différence significative → Mois est une feature pertinente.


In [47]:
# Winsorisation 1% / 99%
# AqiGlobal exclu : on conserve les vraies valeurs pour la détection des alertes
# Winsorisation 1% / 99%
for col in ["AqiGlobal", "PM25", "PM10"]:
    df[col] = winsorize(df[col].fillna(df[col].median()), limits=[0.01, 0.01])
    print(f"{col} winsorisé — nouveau max : {df[col].max():.2f}")

print("AqiGlobal conservé brut — max :", df["AqiGlobal"].max())

AqiGlobal winsorisé — nouveau max : 76.00
PM25 winsorisé — nouveau max : 76.00
PM10 winsorisé — nouveau max : 55.00
AqiGlobal conservé brut — max : 76.0


### Observations - Outliers

**Détection IQR**

AqiGlobal a 34 outliers au-dessus de 89.5. PM25 en a 23 au-dessus de 103.5. PM10 est le plus impacté avec 506 outliers au-dessus de 38 ce seuil bas s'explique par la forte concentration des valeurs entre 1 et 20 dans notre dataset. Les bornes inférieures négatives sont normales car l'IQR peut produire des bornes négatives pour des variables toujours positives.

**Bootstrapping**

Les deux intervalles de confiance à 95% sont très proches. avec outliers [38.82. 39.52] et sans outliers [38.50. 39.12]. La différence est infime (moins de 0.4 points AQI). Les outliers existent mais sont peu influents sur la moyenne globale. La winsorisation reste justifiée pour protéger le modèle des valeurs extrêmes sans distordre la distribution centrale.

**Test de permutation**

p-value à 0.002 soit bien inférieure à 0.05. La différence de l'AQI moyen entre mars et juin est statistiquement significative. Mois est confirmé comme une feature pertinente pour le modèle. Son encodage sin/cos est justifié.

**Winsorisation**

AqiGlobal et PM25 sont bridés à 76. PM10 à 55. Les pics extrêmes (434 pour AqiGlobal) sont réduits à des valeurs cohérentes avec la distribution réelle du dataset. sans suppression de données.

## 5. Feature Engineering

In [48]:
# Encodage cyclique Heure et Mois
df["Heure_sin"] = np.sin(2 * np.pi * df["Heure"] / 24)
df["Heure_cos"] = np.cos(2 * np.pi * df["Heure"] / 24)
df["Mois_sin"] = np.sin(2 * np.pi * df["Mois"] / 12)
df["Mois_cos"] = np.cos(2 * np.pi * df["Mois"] / 12)
df = df.drop(columns=["Heure", "Mois"])
print("Encodage cyclique Heure et Mois : OK")

Encodage cyclique Heure et Mois : OK


In [49]:
# Encodage cyclique direction du vent (variable circulaire 0-360)
df["wind_dir_sin"] = np.sin(2 * np.pi * df["wind_direction_10m"] / 360)
df["wind_dir_cos"] = np.cos(2 * np.pi * df["wind_direction_10m"] / 360)
df = df.drop(columns=["wind_direction_10m"])
print("Encodage cyclique wind_direction_10m : OK")

Encodage cyclique wind_direction_10m : OK


In [50]:
# Binarisation précipitation
df["precipitation_bin"] = (df["precipitation"] > 0).astype(int)
df = df.drop(columns=["precipitation"])
print("Binarisation precipitation : OK")
print(df["precipitation_bin"].value_counts())

Binarisation precipitation : OK
precipitation_bin
0    8811
1    1715
Name: count, dtype: int64


In [51]:
# One-Hot Encoding NomVille
df = pd.get_dummies(df, columns=["NomVille"], prefix="ville")
ville_cols = [c for c in df.columns if c.startswith("ville_")]
print(f"One-Hot NomVille : {len(ville_cols)} colonnes créées")
print(ville_cols)

One-Hot NomVille : 11 colonnes créées
['ville_Bordeaux', 'ville_Franconville', 'ville_Lille', 'ville_Lyon', 'ville_Marseille', 'ville_Nantes', 'ville_Nice', 'ville_Paris', 'ville_Rennes', 'ville_Strasbourg', 'ville_Toulouse']


In [52]:
# Feature temporelle robuste aux trous : AQI_mean_6h
# Pro-Tip : rolling sur index datetime pour gérer les trous correctement
df = df.sort_values(["DateHeure"]).set_index("DateHeure")

# On recrée une colonne NomVille depuis les colonnes One-Hot pour le groupby
df["NomVille_tmp"] = df[ville_cols].idxmax(axis=1).str.replace("ville_", "")

df["AQI_mean_6h"] = df.groupby("NomVille_tmp")["AqiGlobal"].transform(
    lambda x: x.rolling("6h", min_periods=1).mean()
)

df = df.drop(columns=["NomVille_tmp"]).reset_index()
print(f"AQI_mean_6h créé — NULL : {df['AQI_mean_6h'].isnull().sum()}")

AQI_mean_6h créé — NULL : 0


### Observations - Feature Engineering
**Encodages cycliques** Heure, Mois et wind_direction_10m ont été encodés en sin/cos avec succès. Ces trois variables circulaires sont maintenant représentées correctement sans rupture artificielle entre leurs valeurs extrêmes (heure 23 proche de 0. décembre proche de janvier. 359° proche de 0°).

**Binarisation precipitation** 8811 heures sans pluie (84%) contre 1715 heures avec pluie (16%). La distribution confirme que la version binaire est plus informative que la valeur continue qui était écrasée à 0 dans 75% des cas.

**One-Hot NomVille**  11 colonnes créées. une par ville. Chaque ligne a exactement un 1 et dix 0. Aucune relation ordinale artificielle introduite entre les villes.

**AQI_mean_6h**  aucun NULL après création. Le min_periods=1 du rolling assure que même les premières heures d'une ville ont une valeur calculée sur ce qui est disponible. Le groupby par ville garantit qu'on ne mélange pas les données de Paris et Lyon lors du calcul de la fenêtre glissante.

## 6. Sélection finale des features

In [53]:
FEATURES = [
    "Temperature",
    "Humidite",
    "Pression",
    "VitesseVent",
    "Heure_sin",
    "Heure_cos",
    "Mois_sin",
    "Mois_cos",
    "IsWeekend",
    "wind_dir_sin",
    "wind_dir_cos",
    "cloud_cover",
    "precipitation_bin",
    "AQI_mean_6h",
] + ville_cols

TARGET = "AqiGlobal"

# Colonnes exclues et justification
excluded = [
    "PM25",
    "PM10",
    "NO2",
    "O3",  # Composantes de Y : data leakage
    "MeteoStatus",
    "AirStatus",
]  # Colonnes d audit pipeline

df_final = df[FEATURES + [TARGET, "DateHeure"]].dropna(subset=[TARGET])

print(f"Dataset final : {len(df_final)} lignes. {len(FEATURES)} features")
print(f"Features : {FEATURES}")

Dataset final : 10526 lignes. 25 features
Features : ['Temperature', 'Humidite', 'Pression', 'VitesseVent', 'Heure_sin', 'Heure_cos', 'Mois_sin', 'Mois_cos', 'IsWeekend', 'wind_dir_sin', 'wind_dir_cos', 'cloud_cover', 'precipitation_bin', 'AQI_mean_6h', 'ville_Bordeaux', 'ville_Franconville', 'ville_Lille', 'ville_Lyon', 'ville_Marseille', 'ville_Nantes', 'ville_Nice', 'ville_Paris', 'ville_Rennes', 'ville_Strasbourg', 'ville_Toulouse']


### Observations - Sélection des features
- 25 features pour 10 526 lignes. Le dataset est bien dimensionné pour Random Forest et XGBoost. Le ratio lignes/features est de 421. largement suffisant pour éviter l'overfitting.
- Les 14 features météo et temporelles capturent les conditions physiques qui influencent la qualité de l'air. Les 11 colonnes One-Hot NomVille permettent au modèle d'apprendre des comportements spécifiques à chaque ville. AQI_mean_6h apporte la seule dimension temporelle robuste aux trous.
- Aucune feature avec NULL obligatoire dans la liste finale. Le dropna(subset=[TARGET]) n'a supprimé aucune ligne supplémentaire au-delà des 18 déjà retirées à l'étape 3.

## Note importante sur le data leakage

**Le vrai risque de data leakage:**
- PM25. PM10. NO2. O3 sont des indices AQI partiels fournis par AQICN dans le même appel que AqiGlobal. En production. quand on veut prédire AqiGlobal. ces valeurs sont disponibles au même instant. Donc pas de leakage temporel.
- Mais il y a un autre risque : AqiGlobal est calculé par AQICN à partir de PM25. PM10. NO2. O3. Les inclure comme features revient à donner au modèle les composantes du résultat qu'il doit prédire. Le R² serait artificiellement très élevé mais le modèle ne prédirait rien d'utile en production si ces valeurs sont manquantes.
- La vraie question est : en production. est-ce que PM25. PM10. NO2. O3 seront toujours disponibles quand on prédit AqiGlobal ?
- **Réponse:** non. Lyon n'a jamais PM25 ni O3. Si on les inclut comme features obligatoires. le modèle ne peut pas prédire pour Lyon.
Décision finale : on les exclut. La justification correcte n'est pas "data leakage" mais "indisponibilité en production pour certaines villes". Mise à jour du commentaire dans le code :

## 7. Split temporel - TimeSeriesSplit

In [54]:
from sklearn.model_selection import TimeSeriesSplit

df_final = df_final.sort_values("DateHeure")
X = df_final[FEATURES]
y = df_final[TARGET]

tscv = TimeSeriesSplit(n_splits=3)

print("Validation des splits temporels")
for i, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    train_dates = df_final["DateHeure"].iloc[train_idx]
    test_dates = df_final["DateHeure"].iloc[test_idx]
    print(
        f"Split {i} → Train : {train_dates.min().date()} à {train_dates.max().date()} ({len(train_idx)} lignes)"
    )
    print(
        f"          Test  : {test_dates.min().date()} à {test_dates.max().date()} ({len(test_idx)} lignes)"
    )

Validation des splits temporels
Split 1 → Train : 2026-03-29 à 2026-04-19 (2633 lignes)
          Test  : 2026-04-19 à 2026-05-13 (2631 lignes)
Split 2 → Train : 2026-03-29 à 2026-05-13 (5264 lignes)
          Test  : 2026-05-13 à 2026-06-08 (2631 lignes)
Split 3 → Train : 2026-03-29 à 2026-06-08 (7895 lignes)
          Test  : 2026-06-08 à 2026-07-03 (2631 lignes)


### Observations - Split temporel
Les 3 splits sont parfaitement construits et respectent la chronologie. Chaque split train se termine exactement là où le split test commence. sans chevauchement ni fuite de données.
Les splits test ont tous exactement 2631 lignes. ce qui donne une évaluation cohérente et comparable entre les trois splits. Le train grossit progressivement de 2633 à 7895 lignes. ce qui permet d'observer si le modèle s'améliore en voyant plus de données.

**Couverture temporelle :**

Split 1 — le modèle n'a vu que mars et début avril. Il est testé sur la fin avril et début mai. C'est le split le plus difficile car le dataset d'entraînement est le plus petit.

Split 2 — le modèle a vu mars. avril et début mai. Il est testé sur fin mai et début juin. C'est le split intermédiaire.

Split 3 — le modèle a vu mars à début juin. Il est testé sur la fin juin et début juillet. C'est le split le plus réaliste car il correspond à ce qui se passera en production.

Le R carré du Split 3 sera le plus représentatif de la performance réelle du modèle. C'est celui qu'on retiendra pour comparer Random Forest et XGBoost dans le notebook 05.

## 8. Export du dataset final

In [55]:
df_final.to_csv("../data/processed/dataset_final.csv", index=False)
print(f"Dataset final exporté : {len(df_final)} lignes. {len(FEATURES)} features")
print("Fichier : data/processed/dataset_final.csv")

Dataset final exporté : 10526 lignes. 25 features
Fichier : data/processed/dataset_final.csv


## 9. Récapitulatif des décisions

| Feature brute | Transformation | Feature finale | Justification |
|--------------|---------------|---------------|---------------|
| `Temperature` | Aucune | `Temperature` | Distribution proche normale |
| `Humidite` | Aucune | `Humidite` | Bornée 0-100. acceptable |
| `Pression` | Aucune | `Pression` | Distribution stable |
| `VitesseVent` | Aucune | `VitesseVent` | Distribution acceptable |
| `Heure` | Sin/Cos /24 | `Heure_sin`. `Heure_cos` | Variable cyclique 0-23 |
| `Mois` | Sin/Cos /12 | `Mois_sin`. `Mois_cos` | Variable cyclique 1-12 |
| `IsWeekend` | Aucune (déjà bool) | `IsWeekend` | 0/1 déjà exploitable |
| `wind_direction_10m` | Sin/Cos /360 | `wind_dir_sin`. `wind_dir_cos` | Variable circulaire 0-360° |
| `cloud_cover` | Aucune | `cloud_cover` | Distribution acceptable |
| `precipitation` | Binarisation | `precipitation_bin` | 75% à 0. valeur continue peu informative |
| `NomVille` | One-Hot Encoding | `ville_Paris`... | Pas de relation ordinale entre villes |
| `AqiGlobal` (Y) | Winsorisation 1%/99% | `AqiGlobal` | Pics légitimes bridés |
| `AQI_mean_6h` | Rolling 6h sur index datetime | `AQI_mean_6h` | Feature temporelle robuste aux trous |
| `PM25`. `NO2`. `O3`. `PM10` | Exclus | — | Data leakage (composantes de Y) |
| `MeteoStatus`. `AirStatus` | Exclus | — | Colonnes d audit pipeline |
| `AQI_lag_1`. `rolling_mean_24h` | Exclus | — | Trous temporels → trop de NaN |